# 건강 임베딩 GIST 실험 (RunPod A40)

`scripts/train_embedding_gpu.py --loss gist`를 차례로 실행합니다. 학습 코드는 그 스크립트에 있고, 이 노트북은 실행과 결과 확인만 합니다.

- 묻는 것: hard negative 학습(hn)이 짧은 질문에서 B보다 떨어진 이유가 **가짜 오답(false negative)** 때문인가.
  같은 증상을 다루는 상담글이 많아서, 배치 안의 다른 글이 사실은 정답에 가까운데 오답으로 벌을 받았을 수 있습니다.
- 방법: GISTEmbedLoss. 지금 앱의 모델 B가 안내 모델(guide)이 되어, 질문과 자기 정답 글만큼 가까운 배치 안의 글은 오답으로 세지 않습니다.
  온도 0.05(MNRL의 scale 20과 같음)라서 다른 점은 이 가림 하나뿐입니다. 데이터·설정은 s13-e3 재학습과 같습니다(복사본 53개 제외).
- 시드 13 하나 × 3 epoch, 체크포인트 3개. 기록: DagsHub MLflow `health-embedding`의 `ko-sroberta gist`. 체크포인트: 비공개 모델 저장소 `<사용자>/ko-sroberta-dograg-gist`.
- **OCR 노트북이 끝난 뒤에** 실행하세요(같은 GPU를 나눠 쓰면 둘 다 느려집니다). `/workspace/enkoa-practice-deep-learning` 폴더는 건드리지 않습니다.
- 평가(판사 호출)는 로컬에서 합니다: 짧은 질문 dev 40개로 체크포인트를 고르고, 고른 하나만 holdout 80개와 긴 질문으로 B와 비교합니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 저장소 받기와 패키지 설치
%cd /workspace
!rm -rf dograg && git clone -q --depth 1 --branch feat/embedding-gist https://github.com/rndigkwk/dograg.git
%cd /workspace/dograg
# RunPod 이미지의 blinker, cryptography는 debian이 깔아 pip이 지우지 못한다 (mlflow가 새 버전을 요구)
!pip install -q --ignore-installed blinker cryptography
!pip install -q "sentence-transformers[train]>=5.1" mlflow python-dotenv huggingface_hub
!git log --oneline -1
!df -h /workspace | tail -1

In [ ]:
# 토큰: 입력 칸에 붙여 넣습니다. 화면과 노트북 파일에 남지 않습니다.
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("Hugging Face 토큰 (비공개 데이터 읽기 + 모델 저장소 쓰기): ").strip()
os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass("DagsHub 토큰 (MLflow 기록): ").strip()
os.environ["TRAIN_WORK_DIR"] = "/workspace/embedding-gist"
print("토큰 입력됨")

## 1. 확인 실행

작은 부분으로 두 단계만 돌립니다(1~2분). 안내 모델 B도 여기서 처음 받습니다. 업로드하지 않고, MLflow도 pod 안에만 남깁니다.

In [ ]:
!python scripts/train_embedding_gpu.py --loss gist --smoke --no-upload --local-mlflow --seeds 13 --epochs 2 2>&1 | grep -v "it/s]"
!rm -rf /workspace/embedding-gist

## 2. 본 학습

시드 13 × 3 epoch(A40에서 15분 안팎). 끝나면 체크포인트 3개를 모델 저장소에 올리고 DagsHub에 기록합니다. 출력은 `/workspace/gist.log`에도 남습니다.

In [ ]:
!python scripts/train_embedding_gpu.py --loss gist --seeds 13 --epochs 3 2>&1 | grep --line-buffered -v "it/s]" | tee /workspace/gist.log

## 3. 결과

In [ ]:
import json
import pandas as pd

report = json.load(open("/workspace/embedding-gist/upload/results.json"))
print("기준 모델 dev:", report["base"])
table = pd.DataFrame([{"checkpoint": r["name"], "seed": r["seed"], "epoch": r["epoch"], **r["dev"]} for r in report["runs"]])
display(table)
display(table.groupby("epoch")[["accuracy@1", "accuracy@10", "mrr@10"]].agg(["mean", "std"]).round(4))
print("dev로 고른 체크포인트:", report["best_by_dev"])
print(json.dumps({k: v for k, v in report.items() if k != "settings"}))  # 이 줄을 복사해 보내 주세요

## 4. 정리

업로드가 끝난 것을 확인한 뒤 실행합니다. 학습 데이터와 체크포인트, 받아 둔 모델 캐시(OCR 모델 포함)를 지웁니다. OCR 결과 업로드가 끝났는지도 먼저 확인하세요. 그다음 RunPod에서 pod를 멈추고 볼륨을 지웁니다.

In [ ]:
!rm -rf /workspace/embedding-gist /workspace/dograg/data/df.csv /workspace/dograg/data/finetune /workspace/gist.log
!rm -rf ~/.cache/huggingface
for key in ("HF_TOKEN", "MLFLOW_TRACKING_PASSWORD"):
    os.environ.pop(key, None)
print("지웠습니다")